In [ ]:
#| default_exp build

In [ ]:
#| export
from __future__ import annotations
import json, os, shutil, subprocess, sys, time
from fastcore.all import Path

from kavacha.probe import framework_python, is_framework, py_version, running_from
from kavacha.bundle import finish

In [ ]:
#| export
#: Written into the bundle at build; read back to tell a stale install from a current one.
STAMP = 'build.json'
#: The marker that says a build already re-execed, so a bad environment reports rather than forks.
REEXEC = 'KAVACHA_BUILD_VENV'

def run(*args, **kw):
    print('·', ' '.join(str(a) for a in args))
    return subprocess.run([str(a) for a in args], check=True, **kw)

In [ ]:
#| export
def lock_requirements(root, extras=(), out=None):
    """`uv.lock` as a pinned requirements file, or None where uv cannot answer.

    pip resolves `>=` against whatever it finds and keeps whatever it already has, so a build
    environment goes on shipping the versions it was first made with while the repository moves.
    The lock is the tested set, and the bundle should carry that one.
    """
    root = Path(root)
    if not (root/'uv.lock').exists(): return None
    args = ['uv', 'export', '--frozen', '--no-dev', '--no-hashes', '--no-emit-project',
            '--format', 'requirements-txt']
    for e in extras: args += ['--extra', e]
    try: got = subprocess.run(args, cwd=str(root), capture_output=True, text=True, timeout=300)
    except (OSError, subprocess.SubprocessError): return None
    if got.returncode or not got.stdout.strip():
        print(f'· uv export failed; resolving from pyproject instead: {got.stderr.strip()[:200]}')
        return None
    out = Path(out or root/'packaging'/'.app-venv'/'app-requirements.txt')
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(got.stdout)
    return out

In [ ]:
#| export
def build_venv(python, venv, root, extras=(), force=False):
    "A build environment on `python`: a plain `venv`, since `uv venv` links what we are avoiding."
    venv = Path(venv)
    if force: shutil.rmtree(venv, ignore_errors=True)
    exe = venv/'bin'/'python'
    # A build environment left over from another interpreter is not reusable, and reusing one in
    # silence is how an app goes on shipping an old version while the build prints the new one.
    if exe.exists() and py_version(exe) != py_version(python):
        print(f'· rebuilding {venv.name}: it is on {py_version(exe)}, this build wants {py_version(python)}')
        shutil.rmtree(venv, ignore_errors=True)
    if not exe.exists():
        venv.parent.mkdir(parents=True, exist_ok=True)
        run(python, '-m', 'venv', venv)
    run(exe, '-m', 'pip', 'install', '--upgrade', '--quiet', 'pip', 'setuptools', 'wheel')
    spec = f'{root}[{",".join(extras)}]' if extras else str(root)
    if (req := lock_requirements(root, extras, venv/'app-requirements.txt')) is not None:
        run(exe, '-m', 'pip', 'install', '--quiet', '--upgrade', '-r', req)
        run(exe, '-m', 'pip', 'install', '--quiet', '--no-deps', '-e', spec)
    else: run(exe, '-m', 'pip', 'install', '--quiet', '-e', spec)
    return exe

In [ ]:
#| export
def git_stamp(root, version=''):
    "The commit the tree is on and whether it is dirty, or `None` outside a checkout."
    def git(*a):
        r = subprocess.run(['git', *a], cwd=str(root), capture_output=True, text=True, timeout=30)
        return r.stdout.strip() if not r.returncode else None
    try: sha = git('rev-parse', 'HEAD')
    except (OSError, subprocess.SubprocessError): return None
    if not sha: return None
    return {'commit': sha, 'dirty': bool(git('status', '--porcelain', '--untracked-files=no')),
            'version': version, 'built': time.strftime('%Y-%m-%dT%H:%M:%S')}

def stamp_path(bundle):
    "Where the build stamp lives inside `bundle`, on either platform."
    b = Path(bundle)
    return (b/'Contents'/'Resources'/STAMP) if b.suffix == '.app' else (b/STAMP)

def write_stamp(bundle, root, version=''):
    """Record which commit the bundle was built from.

    A bundle is derived from the tree and nothing else compares them, so an app built before a fix
    installs over one built after it and reports the version it always did.
    """
    if (st := git_stamp(root, version)) is None:
        st = {'commit': '', 'dirty': False, 'version': version,
              'built': time.strftime('%Y-%m-%dT%H:%M:%S')}
    p = stamp_path(bundle)
    if p.parent.is_dir(): p.write_text(json.dumps(st, indent=1) + '\n')
    return st

def read_stamp(bundle):
    "The stamp `bundle` was built with, or None when it has none."
    try: return json.loads(stamp_path(bundle).read_text())
    except (OSError, ValueError): return None

In [ ]:
#| export
def check(spec, root, venv=None):
    "What a build would do here, without doing it. Answers on any platform, Linux included."
    root, venv = Path(root), Path(venv or Path(root)/'packaging'/'.app-venv')
    rows = {'platform': sys.platform, 'interpreter': sys.executable,
            'app': spec.name, 'out': str(spec.out(root))}
    if sys.platform not in ('darwin', 'win32'):
        rows['freezer'] = f'none — {sys.platform} builds nothing; macOS and Windows build on their own OS'
    else: rows['freezer'] = 'py2app' if sys.platform == 'darwin' else 'py2exe'
    if sys.platform == 'darwin':
        rows['framework_build'] = is_framework()
        if not rows['framework_build']:
            rows['would_rebuild'] = str(venv)
            rows['on'] = str(framework_python() or '')
    try:
        import webview  # noqa: F401
        rows['pywebview'] = 'installed'
    except ImportError: rows['pywebview'] = 'MISSING'
    rows['running'] = running_from(spec.out(root))
    return rows

In [ ]:
#| export
def build(spec, root, setup_py, venv=None, alias=False, force=False, rebuild_venv=False,
          identity=None):
    """Build `spec` into `root/dist`, re-execing into a framework Python where macOS needs one.

    `setup_py` is the file that calls `setuptools.setup` with the freezer options; it runs in the
    build environment rather than this one, which is the whole reason for the re-exec.
    """
    root, venv = Path(root), Path(venv or Path(root)/'packaging'/'.app-venv')
    out = spec.out(root)
    if not force and (live := running_from(out)):
        raise SystemExit(
            f'{out} is running as pid {", ".join(map(str, live))}. Quit it first, or pass force.\n'
            'A build replaces the standard library the running app imports from, and it fails from '
            'then on with a zip error that says nothing about the build.')
    if sys.platform == 'darwin' and not is_framework():
        if os.environ.get(REEXEC):
            raise SystemExit(f'{sys.executable} is still not a framework build; run `check` to see why')
        if (found := framework_python()) is None:
            raise SystemExit('py2app needs a framework Python, and this one is a standalone build '
                             'whose stdlib extension modules are not files.\n'
                             'Install one and try again:  brew install python@3.12')
        print(f'framework Python: {found}')
        exe = build_venv(found, venv, root, spec.extras, force=rebuild_venv)
        run(exe, sys.argv[0], *(['--alias'] if alias else []), env={**os.environ, REEXEC: '1'})
        return out
    args = [sys.executable, str(setup_py)] + (['py2app'] if sys.platform == 'darwin' else [])
    if alias and sys.platform == 'darwin': args.append('--alias')
    run(*args, cwd=str(root))
    if out.exists() and sys.platform == 'darwin':
        for name, where in finish(out, spec, identity).items(): print(f'  {name}: {where}')
    st = write_stamp(out, root, spec.version)
    print(f"\nbuilt {out}\n  from {st['commit'][:12]}"
          f"{' with uncommitted changes' if st['dirty'] else ''}")
    return out